# Memory System

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) CrewAI roadmap.

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Run a Crew With `memory=True`

With **`memory=True`**, the crew stores distilled facts after tasks and injects relevant recall into later tasks. **`verbose=True`** prints memory activity so you can see retrieval and storage in the console.

In [ ]:
from crewai import Agent, Crew, Process, Task

researcher = Agent(
    role="Research Analyst",
    goal="Gather concise, accurate facts about the topic",
    backstory="You extract factual bullets and name key entities clearly.",
    verbose=True,
)

writer = Agent(
    role="Content Writer",
    goal="Produce a short summary using prior crew context when helpful",
    backstory="You write tight prose and reuse remembered facts instead of re-researching.",
    verbose=True,
)

research_task = Task(
    description="Research {topic}. Output 3–5 bullet facts and mention important names or orgs.",
    expected_output="Bullet list of facts with entities where relevant.",
    agent=researcher,
)

write_task = Task(
    description="Using crew memory and the research, write one short paragraph about {topic}.",
    expected_output="One paragraph.",
    agent=writer,
    context=[research_task],
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

result = crew.kickoff(inputs={"topic": "CrewAI memory system"})
print(result)

## Key Takeaways

- **`memory=True`** turns on crew-level memory so facts flow from early tasks to later ones.
- **`verbose=True`** surfaces memory-related logging during `kickoff()`—watch for recall before tasks and storage after outputs.
- Pair this with persistent **storage** and **embedder** settings when you want the same knowledge available on the **next** run, not only inside this session.